In [ ]:

# =============================================================================
# Google Ads — Silver (Development)
# Bronze : Files/Development/Bronze/Google_ads/
# Silver : Files/Development/Silver/GoogleAds/
# Money  : prefer amount/cost fields; else micros / 1_000_000
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, LongType, DoubleType
from delta.tables import DeltaTable

WORKSPACE_ID = "718e8176-5d40-4a9c-88ff-50ac97ac49ba"
LAKEHOUSE_ID = "981fbe98-2f01-41d8-bf2f-a85e5cd9e2a2"
BASE_PATH = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
BRONZE_PATH = f"{BASE_PATH}/Files/Development/Bronze/Google_ads"
SILVER_PATH = f"{BASE_PATH}/Files/Development/Silver/GoogleAds"
WATERMARK_PATH = f"{SILVER_PATH}/_control/watermark"
# Incremental (default): upsert only; do not wipe history
FULL_REFRESH = False
INCREMENTAL_LOOKBACK_DAYS = 2
INCREMENTAL_MAX_DAYS = 14
print("[RUNTIME]", BRONZE_PATH, "->", SILVER_PATH, "FULL_REFRESH=", FULL_REFRESH)

def read_bronze(folder):
    path = f"{BRONZE_PATH}/{folder}/"
    try:
        df = spark.read.option("header","true").option("multiLine","true").option("quote",'"').option("escape",'"').csv(path)
        print(f"[READ] {folder}: {df.count():,}")
        return df
    except Exception as e:
        print(f"[SKIP] {folder}: {e}")
        return None

def upsert_delta(df, path, keys, label, date_col=None):
    if df is None or df.isEmpty():
        print(f"[SKIP] {label}")
        return "SKIPPED"
    # watermark filter for daily facts
    if date_col is not None and (not FULL_REFRESH) and DeltaTable.isDeltaTable(spark, path):
        wm = spark.read.format("delta").load(path).agg(F.max(F.col(date_col)).alias("m")).collect()[0]["m"]
        if wm is not None:
            df = df.filter(F.col(date_col).isNotNull() & (F.col(date_col) >= F.date_sub(F.lit(wm), int(INCREMENTAL_LOOKBACK_DAYS))))
            st = df.agg(F.min(date_col).alias("mn"), F.max(date_col).alias("mx"), F.count(F.lit(1)).alias("n")).collect()[0]
            print(f"[INCR] {label} wm={wm} rows={st['n']} range={st['mn']}..{st['mx']}")
            if st["n"] == 0:
                print(f"[SKIP] {label}: no new incremental rows")
                return "SKIPPED"
            if st["mn"] is not None and st["mx"] is not None and (st["mx"] - st["mn"]).days > int(INCREMENTAL_MAX_DAYS):
                raise ValueError(
                    f"Incremental batch for {label} spans {(st['mx']-st['mn']).days} days (> {INCREMENTAL_MAX_DAYS}). "
                    "Refusing large backfill. Set FULL_REFRESH=True only intentionally."
                )
    df = df.dropDuplicates(keys)
    cond = " AND ".join([f"t.`{k}` <=> s.`{k}`" for k in keys])
    if (not FULL_REFRESH) and DeltaTable.isDeltaTable(spark, path):
        DeltaTable.forPath(spark, path).alias("t").merge(df.alias("s"), cond).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        mode = "MERGE"
    else:
        df.write.format("delta").mode("overwrite").option("mergeSchema","true").save(path)
        mode = "OVERWRITE"
    print(f"[OK] {label} ({mode}): {spark.read.format('delta').load(path).count():,}")
    return "SUCCESS"

def money(col, amount_path, micros_path):
    return F.coalesce(
        F.get_json_object(col, amount_path).cast(DecimalType(18,6)),
        F.get_json_object(col, micros_path).cast(DecimalType(18,6))/F.lit(1000000)
    ).cast(DecimalType(18,4))

# campaigns
src = read_bronze("google_campaigns")
if src is not None:
    df = (src.filter(F.col("entity_type")=="campaign").select(
        "connector_id","tenant_id","account_id","account_name",
        F.lit("google_ads").alias("platform"),"batch_id",
        F.col("ingestion_time").cast("timestamp"),
        F.col("extraction_start_date").cast("date"), F.col("extraction_end_date").cast("date"),
        F.coalesce(F.get_json_object("raw_json","$.campaign_id"), F.col("entity_id")).alias("campaign_id"),
        F.get_json_object("raw_json","$.campaign_name").alias("campaign_name"),
        F.get_json_object("raw_json","$.status").alias("status"),
        F.get_json_object("raw_json","$.channel_type").alias("channel_type"),
        money(F.col("raw_json"), "$.budget_amount", "$.budget_micros").alias("daily_budget_inr"),
        F.get_json_object("raw_json","$.start_date").cast("date").alias("start_date"),
        F.get_json_object("raw_json","$.end_date").cast("date").alias("end_date"),
        F.current_timestamp().alias("_silver_processed_at"),
    ).filter(F.col("campaign_id").isNotNull()).orderBy(F.col("ingestion_time").desc()).dropDuplicates(["tenant_id","campaign_id"]))
    upsert_delta(df, f"{SILVER_PATH}/silver_google_campaigns", ["tenant_id","campaign_id"], "silver_google_campaigns")

# ad groups
src = read_bronze("google_ad_groups")
if src is not None:
    df = (src.filter(F.col("entity_type")=="ad_group").select(
        "connector_id","tenant_id","account_id","account_name",
        F.lit("google_ads").alias("platform"),"batch_id",
        F.col("ingestion_time").cast("timestamp"),
        F.col("extraction_start_date").cast("date"), F.col("extraction_end_date").cast("date"),
        F.coalesce(F.get_json_object("raw_json","$.adgroup_id"), F.col("entity_id")).alias("adgroup_id"),
        F.coalesce(F.get_json_object("raw_json","$.campaign_id"), F.col("parent_entity_id")).alias("campaign_id"),
        F.get_json_object("raw_json","$.adgroup_name").alias("adgroup_name"),
        F.get_json_object("raw_json","$.status").alias("status"),
        money(F.col("raw_json"), "$.cpc_bid_amount", "$.cpc_bid_micros").alias("cpc_bid_inr"),
        F.current_timestamp().alias("_silver_processed_at"),
    ).filter(F.col("adgroup_id").isNotNull() & F.col("campaign_id").isNotNull())
     .orderBy(F.col("ingestion_time").desc()).dropDuplicates(["tenant_id","adgroup_id"]))
    upsert_delta(df, f"{SILVER_PATH}/silver_google_adgroups", ["tenant_id","adgroup_id"], "silver_google_adgroups")

# ads
src = read_bronze("google_ads")
if src is not None:
    df = (src.filter(F.col("entity_type")=="ad").select(
        "connector_id","tenant_id","account_id","account_name",
        F.lit("google_ads").alias("platform"),"batch_id",
        F.col("ingestion_time").cast("timestamp"),
        F.col("extraction_start_date").cast("date"), F.col("extraction_end_date").cast("date"),
        F.coalesce(F.get_json_object("raw_json","$.ad_id"), F.col("entity_id")).alias("ad_id"),
        F.coalesce(F.get_json_object("raw_json","$.adgroup_id"), F.col("parent_entity_id")).alias("adgroup_id"),
        F.get_json_object("raw_json","$.campaign_id").alias("campaign_id"),
        F.get_json_object("raw_json","$.ad_type").alias("ad_type"),
        F.get_json_object("raw_json","$.status").alias("status"),
        F.get_json_object("raw_json","$.headline").alias("headline"),
        F.get_json_object("raw_json","$.description").alias("description"),
        F.get_json_object("raw_json","$.final_urls").alias("final_urls"),
        F.current_timestamp().alias("_silver_processed_at"),
    ).filter(F.col("ad_id").isNotNull() & F.col("adgroup_id").isNotNull())
     .orderBy(F.col("ingestion_time").desc()).dropDuplicates(["tenant_id","ad_id"]))
    upsert_delta(df, f"{SILVER_PATH}/silver_google_ads", ["tenant_id","ad_id"], "silver_google_ads")


# campaign performance
src = read_bronze("google_campaign_performance")
if src is not None:
    df = (src.filter(F.col("entity_type")=="campaign_performance").select(
        "connector_id","tenant_id","account_id","account_name", F.lit("google_ads").alias("platform"),"batch_id",
        F.col("ingestion_time").cast("timestamp"), F.col("extraction_start_date").cast("date"), F.col("extraction_end_date").cast("date"),
        F.coalesce(F.get_json_object("raw_json","$.campaign_id"), F.col("entity_id")).alias("campaign_id"),
        F.get_json_object("raw_json","$.date").cast("date").alias("date"),
        F.get_json_object("raw_json","$.impressions").cast(LongType()).alias("impressions"),
        F.get_json_object("raw_json","$.clicks").cast(LongType()).alias("clicks"),
        F.get_json_object("raw_json","$.ctr").cast(DoubleType()).alias("ctr"),
        money(F.col("raw_json"), "$.cost", "$.cost_micros").alias("spend_inr"),
        F.get_json_object("raw_json","$.average_cpc").cast(DoubleType()).alias("average_cpc"),
        F.get_json_object("raw_json","$.conversions").cast(DoubleType()).alias("conversions"),
        F.get_json_object("raw_json","$.conversions_value").cast(DoubleType()).alias("conversions_value"),
        F.get_json_object("raw_json","$.cost_per_conversion").cast(DoubleType()).alias("cost_per_conversion"),
        F.get_json_object("raw_json","$.roas").cast(DoubleType()).alias("roas"),
        F.current_timestamp().alias("_silver_processed_at"),
    ).filter(F.col("campaign_id").isNotNull() & F.col("date").isNotNull())
     .orderBy(F.col("ingestion_time").desc()).dropDuplicates(["tenant_id","campaign_id","date"]))
    upsert_delta(df, f"{SILVER_PATH}/silver_google_campaign_performance", ["tenant_id","campaign_id","date"], "silver_google_campaign_performance", date_col="date")

# ad group performance
src = read_bronze("google_ad_group_performance")
if src is not None:
    df = (src.filter(F.col("entity_type")=="ad_group_performance").select(
        "connector_id","tenant_id","account_id","account_name", F.lit("google_ads").alias("platform"),"batch_id",
        F.col("ingestion_time").cast("timestamp"), F.col("extraction_start_date").cast("date"), F.col("extraction_end_date").cast("date"),
        F.coalesce(F.get_json_object("raw_json","$.adgroup_id"), F.col("entity_id")).alias("adgroup_id"),
        F.get_json_object("raw_json","$.campaign_id").alias("campaign_id"),
        F.get_json_object("raw_json","$.date").cast("date").alias("date"),
        F.get_json_object("raw_json","$.impressions").cast(LongType()).alias("impressions"),
        F.get_json_object("raw_json","$.clicks").cast(LongType()).alias("clicks"),
        F.get_json_object("raw_json","$.ctr").cast(DoubleType()).alias("ctr"),
        money(F.col("raw_json"), "$.cost", "$.cost_micros").alias("spend_inr"),
        F.get_json_object("raw_json","$.average_cpc").cast(DoubleType()).alias("average_cpc"),
        F.get_json_object("raw_json","$.conversions").cast(DoubleType()).alias("conversions"),
        F.get_json_object("raw_json","$.conversions_value").cast(DoubleType()).alias("conversions_value"),
        F.get_json_object("raw_json","$.cost_per_conversion").cast(DoubleType()).alias("cost_per_conversion"),
        F.get_json_object("raw_json","$.roas").cast(DoubleType()).alias("roas"),
        F.get_json_object("raw_json","$.engagements").cast(LongType()).alias("engagements"),
        F.get_json_object("raw_json","$.video_views").cast(LongType()).alias("video_views"),
        F.current_timestamp().alias("_silver_processed_at"),
    ).filter(F.col("adgroup_id").isNotNull() & F.col("date").isNotNull())
     .orderBy(F.col("ingestion_time").desc()).dropDuplicates(["tenant_id","adgroup_id","date"]))
    upsert_delta(df, f"{SILVER_PATH}/silver_google_ad_group_performance", ["tenant_id","adgroup_id","date"], "silver_google_ad_group_performance", date_col="date")

# ad performance
src = read_bronze("google_ad_performance")
if src is not None:
    df = (src.filter(F.col("entity_type")=="ad_performance").select(
        "connector_id","tenant_id","account_id","account_name", F.lit("google_ads").alias("platform"),"batch_id",
        F.col("ingestion_time").cast("timestamp"), F.col("extraction_start_date").cast("date"), F.col("extraction_end_date").cast("date"),
        F.coalesce(F.get_json_object("raw_json","$.ad_id"), F.col("entity_id")).alias("ad_id"),
        F.get_json_object("raw_json","$.adgroup_id").alias("adgroup_id"),
        F.get_json_object("raw_json","$.campaign_id").alias("campaign_id"),
        F.get_json_object("raw_json","$.date").cast("date").alias("date"),
        F.get_json_object("raw_json","$.impressions").cast(LongType()).alias("impressions"),
        F.get_json_object("raw_json","$.clicks").cast(LongType()).alias("clicks"),
        F.get_json_object("raw_json","$.ctr").cast(DoubleType()).alias("ctr"),
        money(F.col("raw_json"), "$.cost", "$.cost_micros").alias("spend_inr"),
        F.get_json_object("raw_json","$.average_cpc").cast(DoubleType()).alias("average_cpc"),
        F.get_json_object("raw_json","$.conversions").cast(DoubleType()).alias("conversions"),
        F.get_json_object("raw_json","$.conversions_value").cast(DoubleType()).alias("conversions_value"),
        F.get_json_object("raw_json","$.cost_per_conversion").cast(DoubleType()).alias("cost_per_conversion"),
        F.get_json_object("raw_json","$.roas").cast(DoubleType()).alias("roas"),
        F.current_timestamp().alias("_silver_processed_at"),
    ).filter(F.col("ad_id").isNotNull() & F.col("date").isNotNull())
     .orderBy(F.col("ingestion_time").desc()).dropDuplicates(["tenant_id","ad_id","date"]))
    upsert_delta(df, f"{SILVER_PATH}/silver_google_ad_performance", ["tenant_id","ad_id","date"], "silver_google_ad_performance", date_col="date")

print("========== Google Ads Silver Complete ==========")
for t in ["silver_google_campaigns","silver_google_adgroups","silver_google_ads","silver_google_campaign_performance","silver_google_ad_group_performance","silver_google_ad_performance"]:
    try:
        print(" ✓", t, spark.read.format("delta").load(f"{SILVER_PATH}/{t}").count())
    except Exception as e:
        print(" ✗", t, e)
